# Лекция 8 — Библиотека Pandas. SQL внутри pandas: задачи для самостоятельной работы

**Темы:** Series и DataFrame, отбор столбцов и строк, `.loc`/`.iloc`, логические маски и `query()`, сортировка, `map`/`apply`, `groupby`, пропуски NaN, `merge`/`concat`; SQL-операции средствами pandas: `SELECT`/`WHERE`/`ORDER BY`/`LIMIT`, `CASE` → `np.where`, `NULL` → `NaN`/`fillna`, агрегаты → `.agg`, `GROUP BY`/`HAVING` → `groupby`, `JOIN` → `merge`, `UNION` → `concat`, подзапросы → переменные и `isin`, CTE → `transform`; и работа с SQLite прямо из pandas (`sqlite3` + `pd.read_sql_query`).

- **Задачи 1–10** — простые (базовый pandas + первый SELECT через `pd.read_sql_query`)
- **Задачи 11–20** — средние (pandas посложнее + SQL-операции средствами pandas)
- **Задачи 21–30** — сложные (merge-аналоги JOIN, UNION и подзапросы на pandas, transform вместо CTE, pandas+SQL)

Все решения должны работать **без интернета**: используйте только стандартную библиотеку, `pandas` и `numpy`. База `sqlite3` в памяти (`sqlite3.connect(":memory:")`) нужна только в задачах 10 и 28–30; в остальных работайте с DataFrame прямо в ячейке. Данные для задач маленькие, создавайте их прямо в ячейке (словарь, `INSERT` или `to_sql`).


## Простые задачи


### 1. Создание DataFrame из словаря
Создайте DataFrame из словаря ниже: ключи станут именами столбцов, значения-списки — столбцами. Выведите результат на экран.

```python
data = {
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "subject": ["Математика", "Физика", "Математика", "Информатика", "Физика"],
    "score": [5, 4, 4, 3, 5],
}
```

Проверьте: форма таблицы `(5, 3)`; список столбцов равен `["name", "subject", "score"]`; сумма столбца `score` равна 21.

**Подсказка:** `pd.DataFrame(data)`; длины всех списков-столбцов обязаны совпадать, иначе pandas выдаст ошибку.


### 2. Столбцы, Series и методы head/tail
Для таблицы из задачи 1 (пересоздайте её в своей ячейке) выполните отбор: один столбец `score`, два столбца `name` и `score`, первые две и последняя строки.

Проверьте: `df["score"]` имеет тип `pd.Series`; `df[["name", "score"]]` — это DataFrame формы `(5, 2)`; в `head(2)` имена `["Анна", "Борис"]`; оценка последней строки равна 5.

**Подсказка:** одинарные квадратные скобки дают Series (один столбец), двойные — DataFrame (список столбцов); за «хвост» отвечает `tail()`.


### 3. `.loc` и `.iloc`: по метке или по позиции
Создайте DataFrame со строковым индексом `s1, s2, s3, s4` и столбцами `name`, `age` (значения: Анна 20, Борис 22, Вера 19, Глеб 21). Получите значения двумя способами.

```python
df = pd.DataFrame(
    {"name": ["Анна", "Борис", "Вера", "Глеб"], "age": [20, 22, 19, 21]},
    index=["s1", "s2", "s3", "s4"],
)
```

Проверьте: `df.loc["s2", "name"] == "Борис"`; `df.iloc[2, 1] == 19`; срез `df.loc["s1":"s3"]` содержит 3 строки (правая граница входит), а `df.iloc[0:2]` — ровно 2 строки (правая граница не входит).

**Подсказка:** `.loc` — по меткам (label), `.iloc` — по целым позициям (integer-loc). Срез меток включается обе границы, срез позиций — как в обычном Python.


### 4. Фильтрация логической маской
Дана таблица школьного магазина:

```python
df = pd.DataFrame({
    "product": ["ручка", "тетрадь", "ластик", "рюкзак", "дневник"],
    "qty": [15, 8, 20, 3, 12],
    "price": [25, 40, 15, 1500, 120],
})
```

Составьте маску «куплено не меньше 12 штук» и отберите по ней строки. Посчитайте выручку по отобранным позициям как `(qty * price).sum()`.

Проверьте: маска даёт ровно 3 строки (`ручка`, `ластик`, `дневник`); их суммарная выручка равна 2115.

**Подсказка:** сравнение столбца с константой (`df["qty"] >= 12`) даёт Series из True/False; подставьте её в `df[...]` — получите отфильтрованные строки.


### 5. Фильтрация методом `query()`
Для той же таблицы магазина отфильтруйте строки методом `query()`, не повторяя `df["..."]` в условии: отобрано не меньше 10 штук И цена меньше 100. Затем отфильтруйте по внешней python-переменной и по строковому значению.

Проверьте: `query("qty >= 10 and price < 100")` оставляет `["ручка", "ластик"]`; при `limit = 100` запрос `query("price <= @limit")` даёт множество товаров `{ручка, тетрадь, ластик}`; `query("product == 'ластик'")` возвращает строку с qty = 20.

**Подсказка:** внешние переменные в `query()` доступны через `@имя`; строковое значение внутри условия берите в кавычки, отличные от кавычек самой строки условия.


### 6. Сортировка: `sort_values` по двум ключам
Дана таблица оценок:

```python
df = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "math": [5, 4, 3, 5, 2],
    "rus": [4, 5, 4, 4, 5],
})
```

Отсортируйте по убыванию `math`, а при равных оценках — по возрастанию имени.

Проверьте: порядок имён после сортировки `["Анна", "Глеб", "Борис", "Вера", "Дарья"]`; исходный `df` не изменился (список имён в нём прежний).

**Подсказка:** `sort_values(["math", "name"], ascending=[False, True])` — списки параметров сопоставляются по позиции; метод возвращает новый DataFrame.


### 7. `value_counts` и уникальные значения
Столбец `city` содержит города восьмерых студентов:

```python
df = pd.DataFrame({"city": ["Москва", "Казань", "Москва", "Уфа",
                            "Казань", "Москва", "Уфа", "Самара"]})
```

Посчитайте частоты каждого города и число различных городов.

Проверьте: самый частый город — `Москва` с 3 упоминаниями; `Казань` встречается 2 раза; всего различных городов 4; сумма всех частот равна 8.

**Подсказка:** `df["city"].value_counts()` возвращает Series частот, отсортированную по убыванию; `nunique()` считает различные значения.


### 8. Первый groupby: сводка по группе
Дана таблица студентов с группами (те же данные, что в задаче 1, но со столбцом группы):

```python
df = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "group": ["А", "Б", "А", "Б", "А"],
    "score": [5, 4, 4, 3, 5],
})
```

Сгруппируйте по `group` и посчитайте средний балл каждой группы, а затем количество студентов и максимальный балл.

Проверьте: средний балл группы `А` равен 14/3 (≈ 4.67), группы `Б` — 3.5; в группе `А` три студента с максимальным баллом 5.

**Подсказка:** `df.groupby("group")["score"].mean()`; несколько агрегатов сразу — `df.groupby("group").agg(n=("score", "count"), best=("score", "max"))`.


### 9. NaN: обнаружение, удаление, заполнение
В таблице оценок затесались пропуски:

```python
import numpy as np
df = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "score": [4.0, np.nan, 5.0, np.nan, 3.0],
})
```

Найдите пропущенные значения, удалите строки с ними и попробуйте заполнить пропуски нулями и средним.

Проверьте: в столбце `score` ровно 2 пропуска; `dropna` оставляет 3 строки; среднее по непустым равно 4.0, а после `fillna(0)` — 2.4; после заполнения средним пропусков не остаётся.

**Подсказка:** `isna().sum()` считает пропуски, `dropna(subset=["score"])` удалит строки с NaN только в этом столбце; агрегаты pandas по умолчанию пропускают NaN.


### 10. Первый SELECT через `pd.read_sql_query`
Модуль `sqlite3` входит в стандартную библиотеку, а pandas умеет забирать результат запроса сразу в DataFrame. Создайте базу в памяти, таблицу `students(name TEXT, score INTEGER)` и пять строк (Анна 5, Борис 4, Вера 3, Глеб 5, Дарья 4); затем с помощью `pd.read_sql_query` (не `fetchall`!) загрузите всю таблицу и имена с оценкой не ниже 4.

```python
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE students (name TEXT, score INTEGER)")
```

Проверьте: загрузка `SELECT * FROM students` даёт DataFrame формы `(5, 2)` со столбцами `['name', 'score']`; запрос с `WHERE score >= 4` возвращает имена `['Анна', 'Борис', 'Глеб', 'Дарья']`.

**Подсказка:** `pd.read_sql_query(sql, con)` принимает открытое соединение `sqlite3` и возвращает обычный DataFrame; строки удобнее загружать `executemany("INSERT INTO students VALUES (?, ?)", rows)` — плейсхолдеры `?` защищают от лишних кавычек; по окончании закройте соединение `con.close()`.


## Средние задачи


### 11. `pd.concat`: склейка таблиц вертикально (аналог UNION)
Соберите «годовую» ведомость из двух триместров и посмотрите, что будет с таблицами разной структуры:

```python
q1 = pd.DataFrame({"name": ["Анна", "Борис"], "score": [5, 4]})
q2 = pd.DataFrame({"name": ["Вера", "Глеб"], "score": [3, 5]})
q3 = pd.DataFrame({"name": ["Дарья"], "city": ["Уфа"]})
```

Проверьте: `pd.concat([q1, q2], ignore_index=True)` содержит 4 строки, индекс — `0,1,2,3`, сумма `score` равна 17; склейка `q1` и `q3` даёт столбцы `["name", "score", "city"]` и ровно 3 значения NaN (у кого нет города / оценки).

**Подсказка:** `concat` дописывает строки одной таблицы под другой (как `UNION` в SQL); при несовпадении столбцов недостающие клетки заполняются NaN, а `ignore_index=True` перенумеровывает строки заново.


### 12. `pd.merge`: соединение по ключу (аналог JOIN)
Студенты хранят `group_id`, а названия групп лежат в отдельной таблице:

```python
students = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "group_id": [1, 1, 2, 2, 3],
})
groups = pd.DataFrame({"group_id": [1, 2], "group_name": ["А", "Б"]})
```

Соедините таблицы способами `inner` и `left`.

Проверьте: `inner`-merge даёт 4 строки (Дарья из группы 3 теряется); `left`-merge сохраняет все 5 студентов, ровно у одной строки `group_name` — NaN, и это Дарья.

**Подсказка:** `students.merge(groups, on="group_id", how="left")`; без явного `how` по умолчанию используется `inner` — «потерянные» строки вернуть нельзя, только перезапустить.


### 13. `map` и `apply`: применение функций к данным
Переведите словесные отметки в числовые и посчитайте средний балл по строке:

```python
marks = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб"],
    "mark": ["отлично", "хорошо", "удовлетворительно", "хорошо"],
})
to_num = {"отлично": 5, "хорошо": 4, "удовлетворительно": 3}
grades = pd.DataFrame({"math": [5, 4, 3], "rus": [4, 4, 5]})
```

Проверьте: после `map(to_num)` столбец чисел равен `[5, 4, 3, 4]`; столбец `avg` таблицы `grades`, посчитанный через `apply(lambda row: row.mean(), axis=1)`, равен `[4.5, 4.0, 4.0]`.

**Подсказка:** `map` работает по одному столбцу (Series) и удобно меняет значения по словарю; `apply(..., axis=1)` вызывает функцию для каждой строки целиком — не забудьте выбрать сразу два столбца `grades[["math", "rus"]]`.


### 14. Маски: `isin`, `between`, отрицание `~`
Для таблицы школьного магазина (данные из задачи 4) составьте четыре фильтра разными способами.

Проверьте: `isin(["ручка", "ластик"])` даёт 2 строки; `price.between(20, 100)` — 2 строки (`ручка` и `тетрадь`); отрицание `~(qty > 10)` — 2 строки; составной `(qty >= 10) & (price < 100)` — 2 строки (`ручка`, `ластик`).

**Подсказка:** `isin` — аналог SQL `IN`, `between` — аналог `BETWEEN` (границы включены), `~` инвертирует маску; несколько масок объединяйте операторами `&` и `|` и обязательно оборачивайте каждое условие в скобки.


### 15. Pivot-подобная сводка: groupby по двум ключам и `unstack`
Собрана таблица оценок по предметам:

```python
exam = pd.DataFrame({
    "student": ["Анна", "Анна", "Борис", "Борис", "Вера", "Вера"],
    "subject": ["math", "phys", "math", "phys", "math", "phys"],
    "score": [5, 4, 3, 4, 4, 5],
})
```

Сделайте матрицу «студент × предмет» из средних оценок: группировка по двум ключам и `unstack()`. Отдельно посчитайте по каждому предмету число оценок и средний балл через именованный `agg`.

Проверьте: матрица имеет форму `(3, 2)`; в ней `Борис/math = 3`, `Анна/phys = 4`; по предмету `math` три оценки со средним 4.0.

**Подсказка:** `exam.groupby(["student", "subject"])["score"].mean().unstack()` превращает «длинную» таблицу в широкую; именованные агрегаты: `exam.groupby("subject").agg(count=("score", "count"), avg=("score", "mean"))`.


### 16. Аналоги `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` средствами pandas
Создайте DataFrame о шести сотрудниках:

```python
emp = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья", "Егор"],
    "dept": ["it", "it", "hr", "sales", "hr", "sales"],
    "salary": [120000, 90000, 70000, 80000, 95000, 60000],
})
```

Выполните три отбора — каждый является аналогом SQL-запроса: (а) `SELECT name, salary FROM ... WHERE dept = 'it' ORDER BY salary DESC`; (б) имена сотрудников с зарплатой от 70000 до 100000 по возрастанию зарплаты, не больше трёх (`BETWEEN` + `ORDER BY` + `LIMIT`); (в) все сотрудники не из `it`, у которых отдел `hr` ИЛИ зарплата меньше 70000 (`WHERE ... AND (... OR ...)`).

Проверьте: (а) после отбора столбцов и сортировки получается `[['Анна', 120000], ['Борис', 90000]]`; (б) имена `['Вера', 'Глеб', 'Борис']`; (в) множество имён `{Вера, Дарья, Егор}`.

**Подсказка:** отбор столбцов — `df[["name", "salary"]]`, строк — маской или `query()`; `sort_values("salary", ascending=False)` — это `ORDER BY`, `head(3)` — `LIMIT`; `between(70000, 100000)` включает границы, а смешивая `&` и `|` оборачивайте каждое условие в скобки.


### 17. Классификация через `np.where` вместо SQL `CASE`
В SQL ветвления пишут внутри запроса конструкцией `CASE WHEN ... THEN ... END`. В pandas/numpy то же самое делают вложенным `np.where`. Для каждого студента добавьте столбец-категорию: `школа` (младше 18), `студент` (от 18 до 21 включительно), `выпускник` (старше 21). Затем посчитайте число человек в каждой категории.

```python
students = pd.DataFrame({
    "name": ["Анна", "Борис", "Вера", "Глеб", "Дарья"],
    "age": [20, 17, 22, 16, 18],
})
```

Проверьте: строка Анны получает категорию `студент`; категории распределены так: `школа` — 2, `студент` — 2, `выпускник` — 1.

**Подсказка:** `np.where(cond, a, b)` — это if/else над массивами; для трёх ветвей их вложен: `np.where(age < 18, 'школа', np.where(age <= 21, 'студент', 'выпускник'))`; частоты категорий посчитает `value_counts()`.


### 18. NaN вместо SQL `NULL`: подсчёт, суммы и `fillna`
В pandas роли SQL `NULL` соответствует `NaN`, и агрегаты pandas тоже его игнорируют, но «подставить ноль» нужно уже другим средством. Даны заказы, у части которых сумма неизвестна: (1, 300), (2, NaN), (3, 200), (4, NaN), (5, 500).

Ответьте на вопросы: сколько заказов без суммы? какова суммарная сумма известных заказов? чему равно среднее по известным суммам и среднее, если NaN считать нулём? сколько непустых значений и сколько всего строк?

Проверьте: пропусков ровно 2; `sum()` по известным суммам равна 1000; `mean()` равна 1000/3, а `fillna(0).mean()` — ровно 200; `count()` даёт 3, тогда как `len()` — 5.

**Подсказка:** `isna().sum()` считает пропуски; `sum`/`mean` по умолчанию пропускают NaN (как агрегаты SQL — NULL); `fillna(0)` — pandas-аналог `COALESCE(amount, 0)`; сравнивать с NaN через `==` бессмысленно — сравнение `df["amount"] == np.nan` дадут только False; используйте `isna()`.


### 19. Строка агрегатов:
Таблица школьного магазина (данные задачи 4: товар, количество, цена). Добавьте столбец выручки (`qty * price`) и соберите все метрики одной «строкой отчёта» — именованным Series: число продаж, суммарную выручку, среднюю цену, минимальное и максимальное количество (имена метрик — аналоги алиасов `AS` в SQL). Отдельно найдите товар с максимальной выручкой.

Проверьте: получается `n = 5`, `revenue = 6935`, `avg_price = 340.0`, минимальное количество 3, максимальное 20; лидирует по выручке `('рюкзак', 4500)`.

**Подсказка:** столбец добавьте через `df.assign(revenue=df["qty"] * df["price"])`; строку отчёта соберите как `pd.Series({"n": len(df), "revenue": df["revenue"].sum(), "avg_price": df["price"].mean(), ...})`; «топ-1» — `sort_values("revenue", ascending=False)` + `iloc[0]`; а синтаксис именованных агрегатов `(столбец, функция)` заработает в полную силу в задаче 20 внутри `groupby().agg()`.


### 20. `GROUP BY` и `HAVING` через `groupby` и фильтр-маску
Для таблицы сотрудников из задачи 16 посчитайте по каждому отделу число сотрудников и среднюю зарплату, упорядочив отделы по убыванию средней зарплаты. Затем оставьте только «дорогие» отделы со средней зарплатой больше 90000 — правда, теперь это фильтр по уже посчитанной таблице групп. И отдельно: если сначала выбросить строки отдела `sales`, групп станет две.

Проверьте: групп ровно три, первая строка — `it` (2 сотрудника, средняя зарплата 105000.0); после фильтра остаётся только отдел `it`; без отдела `sales` групп две.

**Подсказка:** `df.groupby("dept").agg(n=("name", "count"), avg_s=("salary", "mean")).sort_values("avg_s", ascending=False)`; SQL `HAVING` — это обычный фильтр по результату группировки (условие на столбцы `n`/`avg_s`), а `WHERE` — фильтр по исходным строкам до `groupby`.


## Сложные задачи


### 21. `merge`: pandas-аналог `INNER JOIN`
Соберите два DataFrame: `customers(id, name)` — Анна, Борис, Вера, Глеб (id 1–4) и `orders(order_id, customer_id, amount)` — (101, 1, 500), (102, 1, 300), (103, 2, 200), (104, 9, NaN). Обратите внимание: заказ 104 ссылается на несуществующего клиента 9.

Соедините таблицы по `customer_id` способом `inner` и отдельно посчитайте число клиентов, у которых есть заказы.

Проверьте: результат содержит ровно 3 строки `[['Анна', 101, 500.0], ['Анна', 102, 300.0], ['Борис', 103, 200.0]]`; сиротский заказ 104 в него не попал; число различных покупателей в результате равно 2 — Вера и Глеб без заказов исчезли из выдачи.

**Подсказка:** ключи с разными именами соединяют так: `orders.merge(customers, left_on="customer_id", right_on="id", how="inner")`; inner тихо вырезает всё без пары; число различных клиентов — `nunique()`; из-за NaN в `amount` этот столбец станет float64 (значения 500.0, 300.0...).


### 22. `how="left"` и поиск «без пары»
Используйте таблицы из задачи 21. Соедините клиентов с заказами левым способом (клиенты — левая таблица); найдите клиентов вообще без заказов; посчитайте число заказов каждого клиента, включая нули.

Проверьте: левое соединение даёт 5 строк (каждый клиент сохранён); «безпарные» строки — ровно `['Вера', 'Глеб']`; число заказов по клиентам `{Анна: 2, Борис: 1, Вера: 0, Глеб: 0}`.

**Подсказка:** `customers.merge(orders, left_on="id", right_on="customer_id", how="left")`; параметр `indicator=True` добавит служебный столбец `_merge` со значениями `left_only` / `both` — это pandas-аналог `WHERE o.id IS NULL`; `.count()` по столбцу `order_id` считает только НЕпустые значения, поэтому клиенты без заказов получают 0.


### 23. `how="right"` и `how="outer"`
Таблицы те же, что в задаче 21. Соедините клиентов и заказы способом `right` (правая таблица — заказы) и способом `outer`.

Проверьте: `right` возвращает 4 строки (все заказы), ровно у одной из них имя клиента — NaN (это сиротский заказ 104); `outer` возвращает 6 строк: ровно 1 строка с NaN имени и ровно 2 строки с NaN заказа.

**Подсказка:** `how="right"` сохраняет правую таблицу целиком — по сути это `how="left"` с переставленными таблицами; `how="outer"` не теряет вообще никого: совпавшие склеиваются, остальные остаются с пустой половиной; проверяйте пустоту через `isna().sum()`.


### 24. `pd.concat` + `drop_duplicates`: аналоги `UNION` и `UNION ALL`
Соберите два DataFrame участников семинаров: `spb(name)` — Анна, Борис, Вера и `msk(name)` — Глеб, Анна, Дарья (Анна была на обоих). Склейте списки по вертикали с дубликатами (аналог `UNION ALL`) и без дубликатов (аналог `UNION`). Отдельно найдите «двойного» участника.

Проверьте: склейка без дедупликации даёт 6 строк, `drop_duplicates()` — 5; имя, которое встретилось в обоих списках, ровно `['Анна']` — это аналог `GROUP BY name HAVING COUNT(*) = 2`.

**Подсказка:** `pd.concat([a, b], ignore_index=True)` — это `UNION ALL`; `drop_duplicates()` — `UNION`; «двойного» участника ищите так: `value_counts()` и отбор имён со счётчиком больше 1.


### 25. Подзапрос в `WHERE`
Таблица студентов: Анна 5, Борис 4, Вера 3, Глеб 2, Дарья 5. В SQL нельзя написать `WHERE score > AVG(score)` — среднее выносят в подзапрос. В pandas проще: сначала посчитайте агрегат в обычную переменную, потом отфильтруйте по ней. Найдите студентов с оценкой строго выше средней и студентов с максимальной оценкой.

Проверьте: среднее равно 3.8; строго выше среднего оказываются ровно 3 студента — `{Анна, Борис, Дарья}`; с максимальной оценкой — пара `{Анна, Дарья}`.

**Подсказка:** `avg = df["score"].mean()`, далее маска `df["score"] > avg` или `df.query("score > @avg")` (внешние переменные в `query()` передаются через `@`); `df["score"] == df["score"].max()` — аналог `WHERE score = (SELECT MAX(score) ...)`.


### 26. `isin` вместо `IN` и промежуточная таблица вместо подзапроса во `FROM`
Используйте таблицы из задачи 21. (а) Найдите имена клиентов, у которых есть хотя бы один заказ, через `isin` (SQL-аналог — `WHERE id IN (SELECT customer_id FROM orders)`). (б) Постройте сводку «клиент — число заказов»: сначала посчитайте её в отдельный DataFrame (аналог подзапроса во `FROM`), затем соедините с клиентами и оставьте клиентов минимум с двумя заказами.

Проверьте: (а) возвращает `{Анна, Борис}`; (б) — ровно `[['Анна', 2]]`.

**Подсказка:** `cust.loc[cust["id"].isin(orders["customer_id"]), "name"]` — это `IN`; агрегат `orders.groupby("customer_id").size()` даёт Series — превратите его в DataFrame связкой `.rename("n").reset_index()`; подзапрос во `FROM` — это, вообще-то, просто переменная с промежуточным DataFrame.


### 27. `transform` вместо CTE: сравнение со средним по группе
Для таблицы сотрудников из задачи 16 найдите сотрудников, чья зарплата выше средней по их же отделу, упорядочив результат по зарплате по возрастанию. В SQL это пишут через `WITH dept_avg AS (...)`; в pandas то же самое делается без промежуточных таблиц — через `groupby(...).transform(...)`.

Проверьте: таких сотрудников ровно трое — `['Глеб', 'Дарья', 'Анна']` с зарплатами `[80000, 95000, 120000]`; Борис, Вера и Егор — ниже средних своих отделов.

**Подсказка:** `emp.groupby("dept")["salary"].transform("mean")` возвращает столбец той же длины, что исходная таблица (для каждой строки — среднее её отдела, аналог CTE, «приклеенный» к строкам); сравните столбец зарплат с этим столбцом напрямую. Альтернатива в лоб: посчитайте промежуточный DataFrame средних по отделам и сделайте `merge` — результат обязан совпасть.


### 28. `pd.read_sql_query`: из SQLite сразу в DataFrame
Создайте в SQLite таблицу `books(id INTEGER, title TEXT, author TEXT, price INTEGER)`: «Мастер и Маргарита» 450, «Преступление и наказание» 380, «Война и мир» 650, «Капитанская дочка» 250, «Тихий Дон» 520. Функцией `pd.read_sql_query` загрузите в датафрейм книги дешевле 500, отсортированные по цене; отдельно загрузите всю таблицу.

Проверьте: отфильтрованный датафрейм имеет форму `(3, 2)`, порядок названий `['Капитанская дочка', 'Преступление и наказание', 'Мастер и Маргарита']`, средняя цена в нём 360; полная загрузка даёт форму `(5, 4)`.

**Подсказка:** `pd.read_sql_query(sql, con)` принимает готовое соединение `sqlite3` и возвращает обычный DataFrame — дальше с ним работают уже pandas-средствами; слева запрос, справа всегда датафрейм.


### 29. pandas против SQL: один показатель двумя способами
Для таблицы продаж школьного магазина (данные задачи 4) посчитайте выручку по каждому товару (`SUM(qty*price)`) двумя способами: SQL-запросом с `GROUP BY` (через `pd.read_sql_query`) и средствами pandas (`assign` + `groupby`). Сверьте два результата между собой и назовите товар-лидер.

Проверьте: оба способа дают одинаковые ряды значений по всем пяти товарам (сверьте после сортировки по имени товара); лидер — `рюкзак` с выручкой 4500; выручка «дневника» — 1440.

**Подсказка:** приведите оба результата к общему виду: SQL-датафрейму — `set_index("product")`, pandas-агрегату — сортировку `sort_index()`, и сравнивайте векторно через `(a == b).all()`; любой SQL-запрос переводится в pandas-операции — это словарь-переводчик из лекции.


### 30. pandas + SQL: два инструмента для одного запроса и ловушка `NULL`
Для таблиц `customers`/`orders` из задачи 21 найдите клиентов минимум с двумя заказами. Сначала решите средствами pandas; затем загрузите те же данные в SQLite-базу в памяти (удобно через `df.to_sql`) и сверьте с SQL-решением — запрос `JOIN` + `GROUP BY` + `HAVING COUNT(*) >= 2` выполняйте функцией `pd.read_sql_query`.

Затем воспроизведите ловушку NULL в обоих мирах: объясните, почему `(orders["amount"] == np.nan).sum()` возвращает 0 и как правильно найти пропуски в pandas; и почему запрос `SELECT id FROM orders WHERE amount = NULL` не находит ничего — замените сравнение на `IS NULL` и проверьте результат.

Проверьте: оба способа (pandas и SQL) возвращают один и тот же результат — ровно `{Анна}`; в pandas NaN находится только через `isna()` (это заказ 104); в SQLite `= NULL` даёт пустой список, а `IS NULL` — одну строку `[(104,)]`.

**Подсказка:** соединение — `con = sqlite3.connect(":memory:")`, таблицы в него загружаются прямо из датафреймов через `df.to_sql("customers", con, index=False)`; в pandas число заказов клиента — `orders.groupby("customer_id").size()`, а клиентов отберите по `isin` по полученным id; сравнение с NULL/NaN через `=` не истинно никогда — только `IS NULL` в SQL и `isna()` в pandas.


## Где потренироваться дальше

- [Pandas: 10 минут, с чего начать](https://pandas.pydata.org/docs/user_guide/10min.html) — официальный тур по библиотеке
- [Тренажёр SQL](https://sql-ex.ru/) — задачи с проверкой, от простых запросов до подзапросов
- [SQL Academy: тренажёр](https://sql-academy.org/ru/trainer) — задачи по JOIN, группировкам и подзапросам
- [Документация SQLite: язык запросов](https://www.sqlite.org/lang.html) — справочник по синтаксису, который исполняет модуль sqlite3
- [Курсы по SQL на Stepik](https://stepik.org/catalog/Search?word=sql) — интерактивные тренажёры на русском
